# Testing transform scripts with pytest

This module is a primer on `pytest` aimed at authors of transform scripts: the read, transform, write (RTW) pipelines and broader ETL flows that move data into a TM1 cube. Readers are assumed to know Python and to have written or maintained at least one script that reads from a source, reshapes the data, and writes it into Sales Plan or a similar cube. Most readers will not have written automated tests before; the focus here is on what a test is, how `pytest` runs it, and the concrete patterns that make ETL code testable.

The mechanics are simple, but the payoff is large. A transform script that ran fine on yesterday's CSV breaks on today's because someone added a new region code, renamed a column, or changed the FX feed format. Without tests, that breakage is found by a human staring at a wrong cube value, sometimes weeks later. With tests, it is found in seconds, before the script ever runs against production data.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. Why test transform scripts
2. The shape of an RTW pipeline
3. Installing and running pytest
4. Anatomy of a test file
5. The `assert` statement and what failure looks like
6. Making transforms testable: pure functions
7. A first end to end transform test
8. Test discovery, naming, and project layout
9. Parametrize: one test, many cases
10. Fixtures
11. Fixture scope and lifetime
12. Sample data: literals, files, and factories
13. Testing the read step
14. Testing the transform step
15. Testing the write step against a fake TM1
16. Mocking with `unittest.mock`
17. The `monkeypatch` fixture
18. Testing exceptions and failure paths
19. `tmp_path` for filesystem outputs
20. Coverage as a guide, not a goal
21. Running the suite in CI
22. Real world testing principles for ETL
23. Common mistakes

---

## 1. Why test transform scripts

A transform script reads from one system, reshapes data, and writes to another. Its correctness is mostly invisible: the script either produces the right cube values, or it produces wrong cube values that look plausible. A wrong total in a Sales Plan view rarely raises an exception; it just sits there, silently miscounted, until someone notices.

Manual testing of an ETL script is reading the output and squinting. It works for the first run on a tiny sample. It does not work when the script has run weekly for two years, the source CSV has gained three new columns, the FX feed has switched providers, and the regions list has doubled. A test suite captures the assumptions the script depends on once and re checks them on every change.

Three concrete benefits show up almost immediately. First, the author can refactor the transform with confidence: if the tests pass after the refactor, the behaviour has not regressed. Second, edge cases that were thought through once stay thought through forever; they live in the test file rather than in the author's head. Third, when something does break in production, reproducing the bug in a test is the first step of fixing it, and the test then prevents the bug from reappearing.

The cost is low. A test for a transform function is usually four or five lines of code, takes milliseconds to run, and needs no TM1 server. The rest of this module is about how to write such tests so the cost stays low.

## 2. The shape of an RTW pipeline

Most transform scripts have the same shape, even when they are dressed up differently. The pipeline reads source data, transforms it into the shape the target system expects, and writes it. A typical Sales Plan loader looks like this:

In [ ]:
# pipeline/sales_plan.py
import pandas as pd
from TM1py import TM1Service

REGION_ALIASES: dict[str, str] = {
    "EU": "Europe",
    "EMEA": "Europe",
    "NA": "North America",
    "LATAM": "Latin America",
    "APAC": "Asia Pacific",
}

def read_sales_csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path, dtype={"period": str, "region": str, "product": str})

def normalize_region(name: str) -> str:
    cleaned = name.strip()
    return REGION_ALIASES.get(cleaned.upper(), cleaned)

def apply_fx(amount: float, currency: str, rates: dict[str, float]) -> float:
    if currency == "EUR":
        return amount
    return amount * rates[currency]

def transform(
    rows: pd.DataFrame,
    rates: dict[str, float],
) -> dict[tuple[str, ...], float]:
    cells: dict[tuple[str, ...], float] = {}
    for row in rows.itertuples(index=False):
        region = normalize_region(row.region)
        amount_eur = apply_fx(row.amount, row.currency, rates)
        key = (row.period, region, row.product, "Working", "Revenue")
        cells[key] = cells.get(key, 0.0) + amount_eur
    return cells

def write_to_cube(tm1: TM1Service, cells: dict[tuple[str, ...], float]) -> None:
    tm1.cells.write_values("Sales Plan", cells)

def run(path: str, rates: dict[str, float], tm1: TM1Service) -> None:
    rows = read_sales_csv(path)
    cells = transform(rows, rates)
    write_to_cube(tm1, cells)

Three small functions plus an orchestrator. This shape is not an accident; it is what the rest of the module assumes, and §6 explains why each step is its own function.

This running example threads through the whole module. Every test below operates on one of these functions or on `run` as a whole.

## 3. Installing and running pytest

`pytest` is a third party package on PyPI. It is installed once into the project's virtual environment:

In [ ]:
%%bash
pip install pytest

The framework is invoked by running the `pytest` command from the project root:

In [ ]:
%%bash
pytest
# ============================= test session starts =============================
# rootdir: /work/sales-plan-loader
# collected 0 items
# ============================ no tests ran in 0.01s ============================

`pytest` walks the current directory, finds files whose names start with `test_` or end with `_test.py`, imports them, and runs every function whose name starts with `test_`. With no tests yet, the run reports zero collected items. The next section writes the first one.

The command takes the path to a single file or directory if a focused run is wanted (`pytest tests/test_transform.py`), and the `-k` flag filters by name substring (`pytest -k normalize` runs only tests whose name contains `normalize`). The `-x` flag stops at the first failure, and `-q` produces a quieter report. These four flags cover most day to day use.

## 4. Anatomy of a test file

A test file is an ordinary Python module. There is no class to inherit from, no decorator to apply by default, no special boilerplate. A test is a top level function whose name begins with `test_`:

In [ ]:
# tests/test_normalize_region.py
from pipeline.sales_plan import normalize_region

def test_normalize_region_maps_eu_to_europe() -> None:
    assert normalize_region("EU") == "Europe"

def test_normalize_region_strips_whitespace() -> None:
    assert normalize_region("  APAC  ") == "Asia Pacific"

def test_normalize_region_passes_unknown_through() -> None:
    assert normalize_region("Antarctica") == "Antarctica"

Running `pytest` now finds three tests and runs them:

```text
tests/test_normalize_region.py ...                                       [100%]
============================== 3 passed in 0.02s ==============================
```

Each dot is one passing test. A failure shows as `F`, an error during collection or fixture setup as `E`, and a skip as `s`. The function names end up in the report, so descriptive names pay off: `test_normalize_region_strips_whitespace` is a one line specification of the behaviour the test guarantees.

A test function takes no positional arguments unless it is using a fixture (§10) or parametrize (§9). It returns nothing. Its job is to set up some inputs, call the code under test, and assert on the result. That is the entire structure.

## 5. The `assert` statement and what failure looks like

`pytest` builds on Python's plain `assert` statement: if the expression is truthy the test passes silently, and if it is falsy the test fails. There is no `assertEqual`, `assertTrue`, or any of the verbose `unittest` API; ordinary Python comparisons are enough.

In [ ]:
def test_apply_fx_converts_usd_to_eur() -> None:
    rates = {"USD": 0.92}
    assert apply_fx(100.0, "USD", rates) == 92.0

def test_apply_fx_passes_eur_through() -> None:
    assert apply_fx(100.0, "EUR", {}) == 100.0

When an assertion fails, `pytest` rewrites the error message to show both sides of the comparison, the values of any variables involved, and the line of code that failed:

```text
FAILED tests/test_fx.py::test_apply_fx_converts_usd_to_eur
    def test_apply_fx_converts_usd_to_eur() -> None:
        rates = {"USD": 0.92}
>       assert apply_fx(100.0, "USD", rates) == 92.0
E       assert 92.00000001 == 92.0
E         + where 92.00000001 = apply_fx(100.0, "USD", {"USD": 0.92})
```

The `>` line is where the assertion failed; the `E` lines are the rewritten message. This rewriting is the main reason `pytest` tests are short: a plain `assert a == b` already produces a diagnostic message as good as any framework call.

For floating point comparisons, exact equality is unreliable; `pytest.approx` covers the rounding tolerance:

In [ ]:
import pytest

def test_apply_fx_within_tolerance() -> None:
    assert apply_fx(100.0, "USD", {"USD": 0.92}) == pytest.approx(92.0)

`approx` is the right call for any computed float, including totals and averages. Equality is fine for integers, strings, and dicts of strings.

## 6. Making transforms testable: pure functions

The single biggest factor in how testable a transform is, is whether its individual steps are pure functions. A pure function takes inputs, returns an output, and does nothing else: it does not read files, does not call out to a server, does not write to disk, does not depend on a global. Given the same inputs, it always produces the same output.

Pure functions are trivial to test because the test does not need a filesystem, a network, or a TM1 connection. It calls the function with literal arguments and asserts on the return value. Compare:

In [ ]:
# Hard to test: reads a file, calls TM1, no return value
def transform_and_write(path: str, tm1: TM1Service) -> None:
    rows = pd.read_csv(path)
    rows["region"] = rows["region"].str.upper().map(REGION_ALIASES)
    cells = {tuple(r): r.amount for r in rows.itertuples()}
    tm1.cells.write_values("Sales Plan", cells)

# Easy to test: each step is pure
def transform(rows: pd.DataFrame, rates: dict[str, float]) -> dict[tuple[str, ...], float]:
    ...

The first version mixes three concerns: reading a file, computing the result, and writing to TM1. A test would have to provide a real CSV file, a real TM1 server, and reach into both to inspect what happened. The second version takes a `DataFrame` and returns a dict; a test passes in a tiny `DataFrame` literal and asserts on the dict.

The orchestrator function (`run` in §2) is the only place where the impure steps meet, and it stays small. Its job is wiring, not logic. The logic lives in `normalize_region`, `apply_fx`, and `transform`, all of which are pure and all of which can be tested without any infrastructure.

This separation is the most important habit in writing testable transform scripts. A function that takes a path and returns nothing is harder to test than one that takes a `DataFrame` and returns a dict, even if the eventual behaviour is identical.

## 7. A first end to end transform test

With the pieces above in place, a test for the `transform` function reads as straightforward Python:

In [ ]:
# tests/test_transform.py
import pandas as pd
from pipeline.sales_plan import transform

def test_transform_normalizes_region_and_converts_currency() -> None:
    rows = pd.DataFrame([
        {"period": "2026-01", "region": "EU", "product": "Widget",
         "amount": 100.0, "currency": "USD"},
        {"period": "2026-01", "region": "EMEA", "product": "Widget",
         "amount": 50.0, "currency": "EUR"},
    ])
    rates = {"USD": 0.92}

    cells = transform(rows, rates)

    # Both rows land on the same cell because EU and EMEA both map to Europe
    assert cells == {
        ("2026-01", "Europe", "Widget", "Working", "Revenue"): 142.0,
    }

Three blocks: arrange (build inputs), act (call the function), assert (check the result). This is the standard shape, sometimes called Arrange Act Assert, and most tests below follow it.

The test runs in milliseconds. There is no CSV file, no TM1 connection, no FX feed. The `DataFrame` literal contains exactly the rows needed to exercise the two behaviours of interest: that `EU` and `EMEA` collapse into `Europe`, and that USD amounts are converted while EUR amounts pass through. The expected dict spells out the answer in full so the diff on failure is precise.

If a future change to `transform` accidentally drops the FX conversion, the test fails with `assert {... 192.0} == {... 142.0}`, naming the wrong number directly. If a future change adds a new `Version` other than `Working`, the tuple key in the expected dict no longer matches and the test fails with a clear diff. The test is short and the failure messages are specific; that combination is the goal.

## 8. Test discovery, naming, and project layout

`pytest` finds tests by walking the directory tree looking for files that match a discovery pattern, then for functions and classes inside those files. The defaults are well chosen and rarely need changing:

- Files named `test_*.py` or `*_test.py`.
- Functions named `test_*`.
- Classes named `Test*` (without an `__init__`); methods inside them named `test_*`.

A typical project layout for a transform script looks like:

```text
sales-plan-loader/
├── pipeline/
│   ├── __init__.py
│   └── sales_plan.py
├── tests/
│   ├── __init__.py
│   ├── conftest.py
│   ├── test_normalize_region.py
│   ├── test_apply_fx.py
│   ├── test_transform.py
│   └── test_run.py
├── pyproject.toml
└── README.md
```

The `pipeline/` package holds the production code. The `tests/` package mirrors it: one test file per source module, sometimes one per function for larger modules. `conftest.py` is a special pytest file that holds shared fixtures; §10 covers it. The `pyproject.toml` is where pytest configuration lives if any is needed:

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-q"
```

`testpaths` tells pytest to start its search in `tests/` (faster on large repos). `addopts` adds default flags to every invocation.

Naming tests is its own small craft. A good test name reads as the specification it enforces: `test_normalize_region_strips_whitespace` is better than `test_region_1`. The reader of a failure report should be able to tell what broke without opening the test file.

## 9. Parametrize: one test, many cases

A normalization function has many small cases: each alias, the strip, the unknown passthrough. Writing one test function per case is repetitive. `pytest.mark.parametrize` runs the same test body once per case:

In [ ]:
import pytest
from pipeline.sales_plan import normalize_region

@pytest.mark.parametrize("raw, expected", [
    ("EU", "Europe"),
    ("EMEA", "Europe"),
    ("eu", "Europe"),
    ("  APAC  ", "Asia Pacific"),
    ("NA", "North America"),
    ("LATAM", "Latin America"),
    ("Antarctica", "Antarctica"),
])
def test_normalize_region(raw: str, expected: str) -> None:
    assert normalize_region(raw) == expected

Pytest reports each case as its own test:

```text
tests/test_normalize_region.py::test_normalize_region[EU-Europe] PASSED
tests/test_normalize_region.py::test_normalize_region[EMEA-Europe] PASSED
tests/test_normalize_region.py::test_normalize_region[eu-Europe] PASSED
tests/test_normalize_region.py::test_normalize_region[  APAC  -Asia Pacific] PASSED
...
```

When a case fails, the report names the specific case (`test_normalize_region[eu-Europe]`), so the failing input is visible without reading the test body.

`parametrize` is the natural fit for any function with a table of cases: aliases, FX rates, validation rules, formatting variants. The first argument is a comma separated string of parameter names; the second is a list of tuples, each providing one case. For named cases, `pytest.param` adds an `id`:

In [ ]:
@pytest.mark.parametrize("raw, expected", [
    pytest.param("EU", "Europe", id="alias-eu"),
    pytest.param("  APAC  ", "Asia Pacific", id="whitespace-apac"),
    pytest.param("Antarctica", "Antarctica", id="unknown-passthrough"),
])
def test_normalize_region(raw: str, expected: str) -> None:
    assert normalize_region(raw) == expected

The custom `id` shows up in the report instead of the value pair, which is helpful when the values are long or noisy.

## 10. Fixtures

A fixture is a function that builds an input for a test. It is declared with `@pytest.fixture` and consumed by listing its name as a parameter of a test function. Pytest matches the parameter name to the fixture name and passes the fixture's return value in.

In [ ]:
# tests/conftest.py
import pytest
import pandas as pd

@pytest.fixture
def fx_rates() -> dict[str, float]:
    return {"USD": 0.92, "GBP": 1.15, "JPY": 0.0061}

@pytest.fixture
def sample_rows() -> pd.DataFrame:
    return pd.DataFrame([
        {"period": "2026-01", "region": "EU", "product": "Widget",
         "amount": 100.0, "currency": "USD"},
        {"period": "2026-01", "region": "APAC", "product": "Widget",
         "amount": 50.0, "currency": "EUR"},
    ])

In [ ]:
# tests/test_transform.py
from pipeline.sales_plan import transform

def test_transform_uses_fx_rates(fx_rates, sample_rows) -> None:
    cells = transform(sample_rows, fx_rates)
    assert cells[("2026-01", "Europe", "Widget", "Working", "Revenue")] == 92.0

def test_transform_passes_eur_through(fx_rates, sample_rows) -> None:
    cells = transform(sample_rows, fx_rates)
    assert cells[("2026-01", "Asia Pacific", "Widget", "Working", "Revenue")] == 50.0

Pytest sees that each test function declares `fx_rates` and `sample_rows` parameters, looks up fixtures of those names, runs them, and passes the results. Each test gets a fresh copy of the data; mutating one test's `sample_rows` does not affect another's.

`conftest.py` is the conventional place for fixtures that more than one test file uses. Pytest discovers it automatically; tests do not need to import from it. A fixture used by only one test file can live at the top of that file.

A fixture can depend on another fixture by listing it as a parameter. This is how complex test setups are composed without ending up in one giant function:

In [ ]:
@pytest.fixture
def transformed_cells(sample_rows, fx_rates) -> dict[tuple[str, ...], float]:
    return transform(sample_rows, fx_rates)

Tests that need the already transformed result depend on `transformed_cells`; tests that need only the raw rows depend on `sample_rows`.

## 11. Fixture scope and lifetime

By default, a fixture runs once per test that uses it. A fresh `DataFrame` is built for every test, which is what is wanted for mutable state. For expensive setup that is genuinely shared, a `scope` argument extends the lifetime:

In [ ]:
@pytest.fixture(scope="session")
def fx_rates() -> dict[str, float]:
    return {"USD": 0.92, "GBP": 1.15, "JPY": 0.0061}

The four scopes, in order of increasing lifetime:

- `function` (default): one instance per test function.
- `class`: one instance per test class.
- `module`: one instance per test file.
- `session`: one instance per `pytest` invocation.

A wider scope is faster but riskier: any test that mutates the shared object affects every later test. The right rule is to use `function` scope unless the setup is genuinely expensive and the value is genuinely immutable. A dict of FX rates is fine at session scope because tests should not be writing to it. A `DataFrame` is fine at session scope only if every test is read only against it, which is hard to guarantee.

A fixture can also clean up after itself by using `yield`:

In [ ]:
@pytest.fixture
def temp_database():
    db = create_test_database()
    yield db
    db.drop()

The code before `yield` runs as setup; the value yielded is what the test receives; the code after `yield` runs as teardown, even if the test fails. This is the same shape as a context manager and is the right structure for any fixture that holds a resource.

Most ETL tests need neither wide scopes nor teardown. The default function scope with a plain `return` is the common case.

## 12. Sample data: literals, files, and factories

Tests need realistic but tiny inputs. Three patterns cover most situations.

The first is the literal: a dict, list, or `DataFrame` constructed inline in the test or fixture. This is what §7 used. It is the right choice when the data is small enough to read at a glance (a handful of rows) and when seeing the data inline makes the test clearer.

In [ ]:
sample_rows = pd.DataFrame([
    {"period": "2026-01", "region": "EU", "product": "Widget",
     "amount": 100.0, "currency": "USD"},
])

The second is the file: a small CSV or JSON checked into the test directory and read by the test:

```text
tests/
├── data/
│   ├── sample_sales.csv
│   └── malformed_sales.csv
```

In [ ]:
from pathlib import Path
import pytest, pandas as pd

DATA_DIR = Path(__file__).parent / "data"

@pytest.fixture
def sample_rows() -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / "sample_sales.csv")

This is appropriate when the data is too large to read inline (twenty rows or more), when the test is specifically about CSV parsing edge cases, or when sharing the same input across many tests. The file is part of the repository and version controlled like any other source.

The third is the factory: a fixture that returns a function, allowing each test to customize the data it needs:

In [ ]:
@pytest.fixture
def make_row():
    def _make(**overrides) -> dict:
        base = {"period": "2026-01", "region": "EU", "product": "Widget",
                "amount": 100.0, "currency": "USD"}
        return {**base, **overrides}
    return _make

def test_transform_handles_jpy(make_row, fx_rates) -> None:
    rows = pd.DataFrame([make_row(currency="JPY", amount=10000.0)])
    cells = transform(rows, fx_rates)
    assert cells[("2026-01", "Europe", "Widget", "Working", "Revenue")] == pytest.approx(61.0)

This is the right shape when many tests need almost the same row but vary one field. The factory keeps the variation visible in the test (`currency="JPY"`) and the irrelevant fields out of sight.

## 13. Testing the read step

`read_sales_csv` is a thin wrapper over `pd.read_csv`. There is little of its own behaviour to test, but two things are worth pinning: the dtype handling and the error case when the file is missing.

In [ ]:
# tests/test_read.py
from pathlib import Path
import pytest
import pandas as pd
from pipeline.sales_plan import read_sales_csv

DATA_DIR = Path(__file__).parent / "data"

def test_read_sales_csv_keeps_period_as_string() -> None:
    df = read_sales_csv(str(DATA_DIR / "sample_sales.csv"))
    # Without dtype={"period": str}, pandas would parse "2026-01" as a date or drop the leading zero
    assert df["period"].dtype == object
    assert df["period"].iloc[0] == "2026-01"

def test_read_sales_csv_raises_on_missing_file(tmp_path) -> None:
    missing = tmp_path / "does_not_exist.csv"
    with pytest.raises(FileNotFoundError):
        read_sales_csv(str(missing))

The first test exercises the only meaningful behaviour `read_sales_csv` adds over `pd.read_csv`: forcing `period` to stay a string. If a future refactor drops the `dtype` argument, the test fails immediately with `assert dtype('<M8[ns]') == object`, naming the regression directly.

The second test uses `pytest.raises`, a context manager that asserts that the wrapped block raises an exception of the given type. If no exception is raised, or a different type is raised, the test fails. `tmp_path` is a builtin pytest fixture covered in §19; it gives the test a guaranteed empty directory so the test does not depend on the working directory.

Tests for the read step are usually thin. The heavy lifting lives in `pandas`; the script's own behaviour is only the small layer of choices it makes on top. Test those choices; do not test `pandas`.

## 14. Testing the transform step

The transform step is where most of the logic lives, and most of the tests. Each behaviour gets its own test, and parametrize handles the table style cases:

In [ ]:
# tests/test_transform.py
import pandas as pd
import pytest
from pipeline.sales_plan import transform

KEY = ("2026-01", "Europe", "Widget", "Working", "Revenue")

def test_transform_aggregates_duplicate_keys(fx_rates) -> None:
    rows = pd.DataFrame([
        {"period": "2026-01", "region": "EU", "product": "Widget",
         "amount": 100.0, "currency": "EUR"},
        {"period": "2026-01", "region": "EMEA", "product": "Widget",
         "amount": 50.0, "currency": "EUR"},
    ])
    cells = transform(rows, fx_rates)
    assert cells[KEY] == 150.0

def test_transform_returns_empty_dict_on_empty_input(fx_rates) -> None:
    rows = pd.DataFrame(columns=["period", "region", "product", "amount", "currency"])
    assert transform(rows, fx_rates) == {}

def test_transform_raises_on_unknown_currency() -> None:
    rows = pd.DataFrame([
        {"period": "2026-01", "region": "EU", "product": "Widget",
         "amount": 100.0, "currency": "ZZZ"},
    ])
    with pytest.raises(KeyError):
        transform(rows, {})

Three distinct behaviours, one test each:

1. Two source rows with the same target key aggregate, not overwrite. This is the most important property of a transform that feeds a cube; getting it wrong silently halves a number.
2. Empty input produces an empty output. This is a typical edge case that the transform happens to handle correctly because of how the loop is written, but the test makes that property explicit and prevents a future change from breaking it.
3. An unknown currency raises rather than silently skipping or producing zero. This pins the contract: the transform fails loudly on bad input, and the script's caller is responsible for handling the failure.

Each test reads as a sentence about behaviour. That is the standard to aim for.

## 15. Testing the write step against a fake TM1

The write step talks to TM1. A unit test cannot afford a real TM1 connection: it would be slow, flaky, and require a server to be running. The standard solution is to substitute a fake object that records the calls without performing them.

`unittest.mock.MagicMock` is the simplest way:

In [ ]:
# tests/test_write.py
from unittest.mock import MagicMock
from pipeline.sales_plan import write_to_cube

def test_write_to_cube_passes_cells_to_tm1py() -> None:
    tm1 = MagicMock()
    cells = {("2026-01", "Europe", "Widget", "Working", "Revenue"): 142.0}

    write_to_cube(tm1, cells)

    tm1.cells.write_values.assert_called_once_with("Sales Plan", cells)

A `MagicMock` returns another `MagicMock` for any attribute access, so `tm1.cells.write_values` is a callable mock that records its arguments. After `write_to_cube` runs, `assert_called_once_with` checks that it was called exactly once and with exactly the given arguments. If the test had passed the wrong cube name, used a different tm1py method, or called it twice, the assertion would fail with a precise message.

The same pattern handles more complicated calls. To check that the script handled an empty `cells` dict by skipping the write entirely:

In [ ]:
def test_write_to_cube_skips_when_cells_empty() -> None:
    tm1 = MagicMock()
    write_to_cube(tm1, {})
    tm1.cells.write_values.assert_not_called()

(This test fails against the §2 implementation, which always calls `write_values`. That is a deliberate signal: the test specifies a behaviour, and the code does not implement it. The fix is to add `if not cells: return` to `write_to_cube`. Tests driving the implementation in this way is one of the everyday uses of a test suite.)

The mock is the test's stand in for the real world. It records what the code under test asked for; the test asserts that the request was correct. Whether the real TM1 server would accept the request is a different question, answered by integration tests run separately.

## 16. Mocking with `unittest.mock`

`MagicMock` is one of three patterns from the `unittest.mock` module that show up in transform tests.

**`MagicMock` for substituting an object.** Whenever a test needs to stand in for a complex object like `TM1Service`, a `MagicMock` is the default. It accepts any method call, returns another `MagicMock`, and remembers everything for inspection.

In [ ]:
tm1 = MagicMock()
tm1.cubes.get_all_names.return_value = ["Sales Plan", "General Ledger"]

names = tm1.cubes.get_all_names()
assert names == ["Sales Plan", "General Ledger"]
tm1.cubes.get_all_names.assert_called_once()

`return_value` configures what a mock method returns. `side_effect` is the same idea for exceptions or callables: setting `side_effect = ConnectionError("nope")` makes the mock raise on call.

**`patch` for replacing a name in a module.** When the code under test imports something at the top of its module and uses it as a global, the test cannot pass a substitute in directly. `patch` replaces the imported name for the duration of the test:

In [ ]:
from unittest.mock import patch
import pipeline.sales_plan as sp

def test_run_uses_csv_path(tmp_path) -> None:
    csv = tmp_path / "in.csv"
    csv.write_text("period,region,product,amount,currency\n2026-01,EU,Widget,100,EUR\n")
    with patch.object(sp, "write_to_cube") as fake_write:
        sp.run(str(csv), {"USD": 0.92}, tm1=MagicMock())
        fake_write.assert_called_once()

`patch.object(sp, "write_to_cube")` replaces `pipeline.sales_plan.write_to_cube` with a `MagicMock` while the `with` block is active and restores the original on exit. The test exercises `run` end to end without performing the real write.

**`patch` as a decorator.** The same operation can decorate a test function instead of using a `with` block:

In [ ]:
@patch("pipeline.sales_plan.write_to_cube")
def test_run_calls_write(fake_write, tmp_path) -> None:
    ...

The patched mock is passed in as an argument. Multiple decorators stack from bottom to top, which is the most common source of confusion when reading patched tests.

The tradeoff with mocks is that they verify call patterns, not actual behaviour. A test that asserts `tm1.cells.write_values.assert_called_once_with("Sales Plan", {...})` is correct only if `write_values` really is the right method. If tm1py renames it, the test still passes against the mock and fails against the real server. Mocks need to be paired with at least one integration test against a real TM1 instance, run less often.

## 17. The `monkeypatch` fixture

`monkeypatch` is pytest's builtin alternative to `patch`. It is a fixture, not a context manager, and it automatically reverses its changes at the end of the test:

In [ ]:
def test_run_reads_fx_rates_from_env(monkeypatch, tmp_path) -> None:
    monkeypatch.setenv("FX_USD_TO_EUR", "0.92")
    csv = tmp_path / "in.csv"
    csv.write_text("period,region,product,amount,currency\n2026-01,EU,Widget,100,USD\n")

    rates_from_env = {"USD": float(os.environ["FX_USD_TO_EUR"])}
    cells = transform(read_sales_csv(str(csv)), rates_from_env)

    assert cells[("2026-01", "Europe", "Widget", "Working", "Revenue")] == 92.0

The setenv change is undone after the test, so the test does not leak state into other tests. `monkeypatch` covers the cases that show up most in ETL code:

- `monkeypatch.setenv("KEY", "value")` and `monkeypatch.delenv("KEY")` for environment variables.
- `monkeypatch.setattr(module, "name", value)` for replacing a name in a module (the same effect as `patch.object`, but undone automatically).
- `monkeypatch.chdir(path)` for changing the working directory.

The advantage over `unittest.mock.patch` is that there is no `with` block and no decorator: the test reads top to bottom, and the cleanup is implicit. The disadvantage is that the change applies for the rest of the test, not for a narrow scope inside it. For most ETL tests, the wider scope is what is wanted.

## 18. Testing exceptions and failure paths

Every transform script has failure paths: a missing file, malformed input, an unreachable TM1 server, a value that does not match a dimension. Tests should pin these explicitly. `pytest.raises` is the primary tool:

In [ ]:
import pytest
from pipeline.sales_plan import apply_fx, read_sales_csv

def test_apply_fx_raises_on_unknown_currency() -> None:
    with pytest.raises(KeyError):
        apply_fx(100.0, "ZZZ", {"USD": 0.92})

def test_apply_fx_error_names_the_currency() -> None:
    with pytest.raises(KeyError, match="ZZZ"):
        apply_fx(100.0, "ZZZ", {"USD": 0.92})

def test_read_sales_csv_raises_filenotfounderror(tmp_path) -> None:
    with pytest.raises(FileNotFoundError):
        read_sales_csv(str(tmp_path / "missing.csv"))

`match` is a regular expression; the test fails unless the exception's `str()` matches it. This is how a test pins both the type of the failure and a useful piece of its message, so a future refactor that hides the offending value behind a generic "validation error" gets caught.

To inspect the exception further, `pytest.raises` returns an `ExceptionInfo` from its context manager:

In [ ]:
def test_apply_fx_carries_currency_in_args() -> None:
    with pytest.raises(KeyError) as exc_info:
        apply_fx(100.0, "ZZZ", {"USD": 0.92})
    assert exc_info.value.args[0] == "ZZZ"

For warnings rather than exceptions, `pytest.warns` is the analogue:

In [ ]:
import warnings, pytest

def test_run_warns_on_partial_rates() -> None:
    with pytest.warns(UserWarning, match="missing FX rate"):
        run_with_partial_rates()

A test suite that covers only the happy path catches half the bugs at most. The failure paths are usually the ones that break in production, because they get less manual exercise. Spend at least as much effort testing them.

## 19. `tmp_path` for filesystem outputs

`tmp_path` is a builtin fixture that gives each test a fresh, empty directory as a `pathlib.Path`. The directory is cleaned up automatically after the test completes. It is the right place to write any file the test produces.

In [ ]:
def test_run_writes_audit_log(tmp_path, monkeypatch) -> None:
    audit = tmp_path / "audit.log"
    monkeypatch.setenv("AUDIT_LOG_PATH", str(audit))

    csv = tmp_path / "in.csv"
    csv.write_text("period,region,product,amount,currency\n2026-01,EU,Widget,100,EUR\n")

    run(str(csv), rates={}, tm1=MagicMock())

    assert audit.exists()
    content = audit.read_text()
    assert "Sales Plan" in content
    assert "1 cell written" in content

Tests that write into the project directory or into `/tmp` directly are a common source of trouble: parallel test runs collide, leftover files affect later runs, and the test depends on the current working directory. `tmp_path` makes all three problems go away. The directory exists for the duration of one test and no other test sees it.

A related fixture, `tmp_path_factory`, is the same idea at session scope. It is appropriate for an expensive directory that many tests share read only access to.

For tests that need a directory laid out a specific way, the fixture composes naturally with the standard library:

In [ ]:
@pytest.fixture
def fake_input_dir(tmp_path) -> Path:
    (tmp_path / "in" / "2026-01").mkdir(parents=True)
    (tmp_path / "in" / "2026-01" / "sales.csv").write_text("...")
    return tmp_path / "in"

Tests then depend on `fake_input_dir` and receive a ready built tree. The teardown is still automatic.

## 20. Coverage as a guide, not a goal

Test coverage measures which lines of production code were executed during the test run. The `coverage.py` package, plugged in as `pytest-cov`, produces a report:

In [ ]:
%%bash
pip install pytest-cov
pytest --cov=pipeline --cov-report=term-missing
# Name                       Stmts   Miss  Cover   Missing
# --------------------------------------------------------
# pipeline/__init__.py           0      0   100%
# pipeline/sales_plan.py        38      4    89%   42-45

The `Missing` column lists line numbers that no test touched. Reading the report is a quick way to find untested code: a line that has never run is a line whose behaviour is not pinned, and the next change to it might break silently.

Coverage is a useful diagnostic but a misleading target. A line of code can be executed by a test that does not actually assert anything meaningful about it, so 100% line coverage does not mean the code is well tested. A test that calls every function but never checks the results can hit 100%. The number is best read as "below this, definitely undertested" rather than "above this, definitely well tested."

Two practical guidelines. First, treat any uncovered branch of a transform as a yellow flag: it represents a behaviour the script does in production that no test ever exercises. Either add the test or remove the branch. Second, do not chase the last few percent; the right test is often a few lines of error handling that is hard to trigger from a unit test and is better covered by an integration test or by code review.

A reasonable target for a transform script is the high 80s or low 90s line coverage, with a focus on covering every distinct behaviour rather than every distinct line. The transform function (§14) deserves higher coverage than the orchestrator, because the transform is where the logic lives.

## 21. Running the suite in CI

A test suite that only runs locally is half a test suite. The other half is running it on every push, against every pull request, before any merge. GitHub Actions is the most common host; the configuration is short:

```yaml
# .github/workflows/test.yml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -e . pytest pytest-cov
      - run: pytest --cov=pipeline --cov-report=term-missing
```

The same shape works on GitLab CI, Azure Pipelines, and Jenkins. The essentials are: install the project and `pytest`, run `pytest`, fail the build if any test fails. The CI run is then the gate that prevents broken transforms from reaching `main`.

For tests that need a real TM1 instance, CI introduces a complication: there has to be one available. Common solutions are a dedicated test instance running in a container started by the CI job, or an integration suite that runs only on a schedule against a long lived test environment. Either way, the integration tests live in a separate file or a separate marker (`@pytest.mark.integration`) so the fast unit tests can run on every push without needing TM1.

Marking tests is a one liner:

In [ ]:
@pytest.mark.integration
def test_full_round_trip_against_real_tm1(tm1_service) -> None:
    ...

The default `pytest` invocation skips integration tests with a deselect (`pytest -m "not integration"`); a separate scheduled job runs them with `pytest -m integration`. The unit suite runs in seconds; the integration suite runs in minutes; both are visible to the team.

## 22. Real world testing principles for ETL

Once the mechanics are familiar, the practical question is what to test, in what order, and how much. A handful of guidelines apply.

**Test the transform first, the orchestrator last.** The transform function carries the logic; the orchestrator carries the wiring. Bugs in the transform are silent and produce wrong cube values. Bugs in the orchestrator are loud and surface immediately on the first run. Investing test effort proportionally to the impact of failure means the transform gets thorough coverage and the orchestrator gets a smoke test or two.

**One behaviour per test.** A test named `test_transform` that aggregates duplicates, applies FX, normalizes regions, and validates dimensions is impossible to read on failure: the report says it failed, not which behaviour broke. Splitting it into four named tests makes the failure point obvious. The cost of more test functions is small; the benefit of clear failure messages is large.

**Pin the contract, not the implementation.** Tests that check internal call sequences (`first this method runs, then that one`) break on every refactor even when the behaviour is unchanged. Tests that check the output of the transform stay valid through any refactor that preserves the behaviour. Mocks invite implementation pinning; resist them. An assertion on a returned dict is more durable than an assertion on which methods were called in which order.

**Test the failure paths, not just the happy path.** Almost every production incident comes from a path that no manual test ever exercised: the malformed row, the missing column, the unknown currency, the network timeout. These paths are cheap to test (`pytest.raises` plus a literal input), and the test is the documentation of how the script is supposed to fail.

**Keep tests fast.** A unit test that takes more than fifty milliseconds is a candidate for redesign; a suite that takes more than a minute discourages running it. The way to keep tests fast is to keep the inputs tiny (a few rows, not a few thousand) and to mock the slow parts (TM1, the network, large files). A slow test does not run as often and therefore does not catch as many bugs.

**Tests are read more often than they are written.** Every test will be read by someone debugging a failure, possibly years later. Optimize the test for that reader: a clear name, a small input, a precise assertion, a single behaviour. Cleverness in tests is a bug.

**Run them on every change.** A test suite that runs on a developer's laptop once a week catches some bugs, occasionally. A suite that runs in CI on every push catches almost all of them, immediately. The mechanics in §21 are the difference between a suite that pays for itself and one that does not.

## 23. Common mistakes

A short collection of errors that show up in nearly every first attempt at a transform test suite.

**Testing against a real TM1 server in unit tests.** A unit test that calls a real `TM1Service` is slow, flaky, and depends on the server being in a known state. The right pattern is a `MagicMock` substitute (§15). Real TM1 calls belong in a separate integration suite, run on a schedule, not on every push.

In [ ]:
# Wrong
def test_write_to_cube() -> None:
    tm1 = TM1Service(address="real.tm1.example.com", ...)
    write_to_cube(tm1, {("2026-01", "Europe", "Widget", "Working", "Revenue"): 100.0})
    # No assertion, no cleanup, depends on network

In [ ]:
# Correct
def test_write_to_cube() -> None:
    tm1 = MagicMock()
    write_to_cube(tm1, {("2026-01", "Europe", "Widget", "Working", "Revenue"): 100.0})
    tm1.cells.write_values.assert_called_once_with(
        "Sales Plan",
        {("2026-01", "Europe", "Widget", "Working", "Revenue"): 100.0},
    )

**Asserting on the wrong thing.** A test that calls the transform and checks `len(cells) == 1` fails to catch the most common bug: the transform produced one cell but with the wrong key or wrong value. Assert on the full result, not on shape proxies.

In [ ]:
# Wrong
def test_transform_produces_a_cell(sample_rows, fx_rates) -> None:
    cells = transform(sample_rows, fx_rates)
    assert len(cells) == 1

In [ ]:
# Correct
def test_transform_produces_expected_cell(sample_rows, fx_rates) -> None:
    cells = transform(sample_rows, fx_rates)
    assert cells == {
        ("2026-01", "Europe", "Widget", "Working", "Revenue"): 92.0,
        ("2026-01", "Asia Pacific", "Widget", "Working", "Revenue"): 50.0,
    }

**Sharing mutable state across tests.** A module level dict or a session scoped fixture that the test mutates leaks state into later tests, producing failures that depend on test order.

In [ ]:
# Wrong
RATES = {"USD": 0.92}

def test_eur_passthrough() -> None:
    RATES["EUR"] = 1.0      # mutates global; next test sees the change
    assert apply_fx(100.0, "EUR", RATES) == 100.0

In [ ]:
# Correct
@pytest.fixture
def rates() -> dict[str, float]:
    return {"USD": 0.92}    # fresh dict per test

def test_eur_passthrough(rates) -> None:
    assert apply_fx(100.0, "EUR", rates) == 100.0

**Catching exceptions instead of asserting on them.** Wrapping the call in `try`/`except` to "test for a failure" silently passes when no exception is raised, which is the opposite of what is wanted.

In [ ]:
# Wrong
def test_apply_fx_unknown_currency() -> None:
    try:
        apply_fx(100.0, "ZZZ", {"USD": 0.92})
    except KeyError:
        pass
    # Test passes whether or not KeyError was raised

In [ ]:
# Correct
def test_apply_fx_unknown_currency() -> None:
    with pytest.raises(KeyError):
        apply_fx(100.0, "ZZZ", {"USD": 0.92})

**Testing through the orchestrator.** Running the entire `run` function and asserting on the final cube state mixes every concern into one test. When it fails, there is no way to tell whether the read, the transform, or the write broke. Test each function on its own; reserve end to end tests for one or two smoke checks.

In [ ]:
# Wrong: one large test that hides which step failed
def test_pipeline(tmp_path) -> None:
    csv = tmp_path / "in.csv"
    csv.write_text(...)
    tm1 = MagicMock()
    run(str(csv), {"USD": 0.92}, tm1)
    tm1.cells.write_values.assert_called_once_with("Sales Plan", {...})

In [ ]:
# Correct: separate tests for separate behaviours
def test_read_sales_csv_keeps_period_as_string() -> None: ...
def test_transform_aggregates_duplicate_keys() -> None: ...
def test_write_to_cube_passes_cells_to_tm1py() -> None: ...
def test_run_smoke_test() -> None: ...   # one end to end, just to wire it together

**Floating point equality.** A computed float compared with `==` is a flaky test waiting to happen, especially after FX conversions and aggregations.

In [ ]:
# Wrong
assert apply_fx(100.0, "USD", {"USD": 0.92}) == 92.0

In [ ]:
# Correct
assert apply_fx(100.0, "USD", {"USD": 0.92}) == pytest.approx(92.0)

**Tests that depend on the working directory.** Reading or writing a relative path inside a test makes the test pass from one directory and fail from another. `tmp_path` and `Path(__file__).parent` solve this; relative paths in tests do not.

In [ ]:
# Wrong
df = pd.read_csv("tests/data/sample_sales.csv")   # depends on CWD

In [ ]:
# Correct
DATA_DIR = Path(__file__).parent / "data"
df = pd.read_csv(DATA_DIR / "sample_sales.csv")